
## dedupe_text_chunks.ipynb

## Description
Content-based deduplication for text chunk datasets.  
This notebook scans a folder recursively for `.txt` files, computes a **SHA-256 hash** of each file's content (using chunked reads for efficiency), and deletes any duplicates found. When a duplicate `.txt` is removed, the notebook also attempts to delete its paired `.metadata.json` file if present.

---

### Core Functions
- **`file_hash(path)`**  
    Computes a SHA-256 digest of the file at `path` using 8KB chunked reads to minimize memory usage. Returns the hex-encoded digest string.

- **`remove_duplicates(folder_path, dry_run=True)`**  
    Iterates all `**/*.txt` under `folder_path`, builds a map of seen content hashes to the first-seen file, and removes subsequent duplicates.  
    Also removes the paired `.metadata.json` for a deleted `.txt` when found.  
    Returns a list of file paths that were (or would be) deleted.

---

### Duplicate Definition
Two `.txt` files are considered duplicates if their **SHA-256 hashes match**.  
(This checks file content, not file names.)

---

### Paired Metadata Convention
- For a text file:  /path/to/foo.txt
- for a metadata file: /path/to/foo.metadata.json


In [ ]:
import os
import hashlib
from pathlib import Path

In [ ]:
def file_hash(path):

    """
    Compute a SHA-256 hash of a file using chunked reads to avoid high memory usage.

    Args:
        path (Path): Path to the file to hash.

    Returns:
        str: Hex-encoded SHA-256 digest of the file contents.
    """

    h= hashlib.sha256()
    with open(path, "rb") as f:
        # Read the file in 8KB chunks to handle large files efficiently
        for chunk in iter(lambda:f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def remove_duplicates(folder_path):

    """
    Remove duplicate text chunks based on the *content* of their `.txt` files.
    When a duplicate `.txt` is deleted, the function also deletes the paired
    `.metadata.json` file if it exists.

    Duplicate definition:
        Two `.txt` files are considered duplicates if their SHA-256 hashes match.

    Matching metadata:
        Given a text file like: `.../foo.txt`
        The paired metadata is expected at: `.../foo.metadata.json`

    Args:
        folder_path (str | Path): Root folder to scan recursively.

    Returns:
        List[Path]: A list of file paths that were deleted (both .txt and .metadata.json).
    """

    folder = Path(folder_path)
    seen_hashes = {} # content hash -> first seen .txt Path
    deleted_files = []

    # Recursively iterate over all .txt files under the folder
    for txt_file in folder.glob("**/*.txt"):
        # Compute content hash of the current text file
        h = file_hash(txt_file)

        # Build the expected .metadata.json path for this text file
        # Example: /path/foo.txt -> base=/path/foo -> meta=/path/foo.metadata.json
        base = txt_file.with_suffix("")  # remove .txt
        meta_file = base.with_suffix(".txt.metadata.json") # add .metadata.json

        if h in seen_hashes:
            # This content was already seen -> duplicate detected
            os.remove(txt_file)
            deleted_files.append(txt_file)
            # If there is a paired metadata file, delete it as well
            if meta_file.exists():
                os.remove(meta_file)
                deleted_files.append(meta_file)
                # Informational log (keep friendly & concise)
            print(f"Deleted duplicate: {txt_file} & {meta_file}")
        else:
            # First time seeing this content -> remember it
            seen_hashes[h] = txt_file
    
    print(f"Deleted {len(deleted_files)} duplicate files")
    return deleted_files

In [ ]:
# run to remove the duplicates in the given folder
remove_duplicates("../data/text_chunks")